# UMABC vs ABC — Complete Class Evaluation A
## Std functions(Yao f1–f13 + CEC 2005 f14–f27) ·  CEC 2013 (CEC 2013 f28–f55)

**Backend officiel : [opfunu](https://github.com/thieu1995/opfunu)** — shifts/rotations officiels des compétitions CEC.

| Critère | Valeur |
|---------|--------|
| Algorithmes | **ABC** (Karaboga 2005) vs **UMABC** (proposé) |
| Std functions | f1–f13 Yao 1999 + f14–f27 CEC 2005 (opfunu) |
| CEC 2013 | f28–f55 CEC 2013 (opfunu) |
| D | **30** (modifiable Cell 2) |
| Runs | **30** independants |
| MaxFEs | **5 000 × D** |
| Métriques | **Mean ± Std** erreur, Best, Worst, Median |
| Tests stat. | Wilcoxon signed-rank α=0.05 + Friedman |
| Sauvegarde | **Automatique toutes les 30 min** (pickle + CSV) |
| Figures | 300 DPI, Times New Roman,  |


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 0 — Dependancy Installation
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys
pkgs = ['opfunu', 'numpy', 'scipy', 'pandas', 'tabulate', 'matplotlib']
subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'])
import opfunu
print(f'✓ opfunu {opfunu.__version__} — shifts/rotations CEC 2005/2013 officiels')
print('✓ Dépendances prêtes.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — Imports & style publication 
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon, rankdata
from tabulate import tabulate
import warnings, time, os, pickle, glob, threading
import opfunu.cec_based.cec2013 as cec2013
import opfunu.cec_based.cec2005 as cec2005
warnings.filterwarnings('ignore')

# ── Style publication  ───────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'serif',
    'font.serif'     : ['Times New Roman', 'DejaVu Serif'],
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
    'grid.linestyle' : '--',
})

C_ABC   = '#3266AD'   # bleu 
C_UMABC = '#0F6E56'   # vert
C_TIE   = '#888780'   # gris

os.makedirs('results_ABC_UMABC', exist_ok=True)
os.makedirs('results_ABC_UMABC/checkpoints', exist_ok=True)
print('✓ Répertoires : results_ABC_UMABC/  et  results_ABC_UMABC/checkpoints/')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — Experimental Parameters (edit here)
# ═══════════════════════════════════════════════════════════════════
D = 30   # Dimension : 30 | 50 

PARAMS = dict(
    D          = D,
    SN         = D,           # Taille colonie = D  (setting NABC paper Sect.5.1)
    MaxFEs     = 5_000 * D,   # Budget CEC — Tables 1 & 11
    limit      = 50,          # Limite abandon
    # UMABC spécifique
    alpha      = 0.6,         # Balance utilité Eq.(1)
    HUFS_ratio = 0.2,         # Top 20% = HUFS
    # Protocole Classe A
    RUNS       = 30,
    AUTOSAVE_MIN = 30,        # Sauvegarde auto toutes les N minutes
)

print('Paramètres expérimentaux :')
for k, v in PARAMS.items():
    print(f'  {k:20s} = {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Robust Automatic Save System
# ─ Automatic resume at startup (last detected checkpoint)
# ─ Daemon thread: pickle + CSV every 30 min
# ─ Manual resume: load_checkpoint(path) before Cell 9
# ═══════════════════════════════════════════════════════════════════

import os, glob, pickle, threading, time
import numpy as np
import pandas as pd

CKPT_DIR = 'results_ABC_UMABC/checkpoints'
os.makedirs('results_ABC_UMABC', exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Dictionnaire global des résultats ──────────────────────────────
results = {}   # fid -> dict  (partagé entre autosave et experiment)
_autosave_active = [False]


# ═══ Persistance ════════════════════════════════════════════════════

def save_checkpoint(tag='auto'):
    """Sauvegarde pickle + CSV partiel."""
    ts   = time.strftime('%Y%m%d_%H%M%S')
    path = os.path.join(CKPT_DIR, f'chk_{tag}_{ts}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    _export_csv(f'{tag}_{ts}')
    return path

def _export_csv(ts):
    """Export CSV partiel — format CEC officiel."""
    rows = []
    for fid, r in results.items():
        for algo, pfx in [('ABC','abc'), ('UMABC','umabc')]:
            if pfx+'_err' not in r or len(r[pfx+'_err']) == 0:
                continue
            e = r[pfx+'_err']
            rows.append({
                'fid': fid, 'name': r.get('name',''), 'algo': algo,
                'runs_done' : len(e),
                'mean_err'  : float(np.mean(e)),
                'std_err'   : float(np.std(e, ddof=0)),
                'best_err'  : float(np.min(e)),
                'worst_err' : float(np.max(e)),
                'median_err': float(np.median(e)),
            })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(CKPT_DIR, f'partial_{ts}.csv'), index=False)

def load_checkpoint(path=None):
    """
    Charge un checkpoint dans `results`.
    Si path=None, charge automatiquement le plus récent.
    """
    global results
    if path is None:
        chks = sorted(glob.glob(os.path.join(CKPT_DIR, 'chk_*.pkl')))
        if not chks:
            print('Aucun checkpoint trouvé — démarrage à zéro.')
            return
        path = chks[-1]
    with open(path, 'rb') as f:
        data = pickle.load(f)
    results.update(data)
    fns_done = [fid for fid, r in results.items()
                if len(r.get('abc_err', [])) >= PARAMS.get('RUNS', 30)]
    print(f'✓ Checkpoint chargé : {path}')
    print(f'  {len(results)} fonctions en mémoire, '
          f'{len(fns_done)} complètes ({PARAMS.get("RUNS",30)} runs).')
    return results

def manual_save(tag='manual'):
    """Sauvegarde immédiate (appeler à tout moment)."""
    path = save_checkpoint(tag)
    print(f'✓ Sauvegarde manuelle → {path}')
    return path

def list_checkpoints():
    """Affiche les checkpoints disponibles pour reprise."""
    chks = sorted(glob.glob(os.path.join(CKPT_DIR, 'chk_*.pkl')))
    if not chks:
        print('Aucun checkpoint disponible.')
        return []
    print(f'{len(chks)} checkpoint(s) disponibles :')
    for p in chks:
        sz = os.path.getsize(p) / 1024
        ts_str = os.path.basename(p)
        # Charger juste les clés
        try:
            with open(p,'rb') as f: d = pickle.load(f)
            fns_ok = sum(1 for r in d.values()
                         if len(r.get('abc_err',[])) >= PARAMS.get('RUNS',30))
            print(f'  {p}  ({sz:.0f} Ko)  — {len(d)} fns, {fns_ok} complètes')
        except Exception as e:
            print(f'  {p}  ({sz:.0f} Ko)  — [lecture échouée : {e}]')
    return chks


# ═══ Thread autosave ════════════════════════════════════════════════

def _autosave_loop(interval_sec):
    count = 0
    while _autosave_active[0]:
        time.sleep(interval_sec)
        if not _autosave_active[0]: break
        count += 1
        if not results: continue
        try:
            path = save_checkpoint('auto')
            fns_done = sum(1 for r in results.values()
                           if len(r.get('abc_err',[])) >= PARAMS.get('RUNS',30))
            print(f'  [Autosave #{count}] {len(results)} fns / {fns_done} complètes → {path}')
            # Nettoyer les vieux checkpoints 'auto' (garder 5 derniers)
            autos = sorted(glob.glob(os.path.join(CKPT_DIR, 'chk_auto_*.pkl')))
            for old in autos[:-5]:
                try: os.remove(old)
                except: pass
        except Exception as e:
            print(f'  [Autosave #{count}] ERREUR : {e}')

def start_autosave():
    _autosave_active[0] = True
    t = threading.Thread(
        target=_autosave_loop,
        args=(PARAMS['AUTOSAVE_MIN'] * 60,),
        daemon=True)
    t.start()
    print(f'✓ Autosave démarré — intervalle : {PARAMS["AUTOSAVE_MIN"]} min')
    return t

def stop_autosave():
    _autosave_active[0] = False
    print('✓ Autosave arrêté.')


# ═══ Reprise automatique au démarrage ══════════════════════════════

_existing = sorted(glob.glob(os.path.join(CKPT_DIR, 'chk_*.pkl')))
if _existing:
    print(f'⚠ Checkpoints détectés ({len(_existing)}) — reprise automatique du plus récent.')
    load_checkpoint(_existing[-1])
    print()
    print('  Pour repartir de zéro : results.clear()')
    print('  Pour choisir un autre checkpoint : load_checkpoint("chemin/chk_xxx.pkl")')
    print('  Pour voir tous les checkpoints : list_checkpoints()')
else:
    print('✓ Aucun checkpoint existant — démarrage à zéro.')
    print()
    print('  Après une interruption, relancez toutes les cellules :\'')
    print('  le checkpoint sera rechargé automatiquement.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Functions Yao et al. 1999 (Table 1 : f1–f13)
# Exact native implementation — compliant Yao, Liu & Lin (1999)
# ═══════════════════════════════════════════════════════════════════

def _u(x, a=10, k=100, m=4):
    return np.where(x > a, k*(x-a)**m,
           np.where(x < -a, k*(-x-a)**m, 0.0))

def yao_eval(fid, x):
    D = len(x)
    if   fid == 'f1':  return float(np.sum(x**2))
    elif fid == 'f2':  return float(np.sum(np.abs(x)) + np.prod(np.abs(x)))
    elif fid == 'f3':  return float(sum(np.sum(x[:i+1])**2 for i in range(D)))
    elif fid == 'f4':  return float(np.max(np.abs(x)))
    elif fid == 'f5':  return float(np.sum(100*(x[1:]-x[:-1]**2)**2+(x[:-1]-1)**2))
    elif fid == 'f6':  return float(np.sum(np.floor(x+0.5)**2))
    elif fid == 'f7':  return float(np.sum(np.arange(1,D+1)*x**4))+np.random.rand()
    elif fid == 'f8':  return float(-np.sum(x*np.sin(np.sqrt(np.abs(x)))))
    elif fid == 'f9':  return float(10*D+np.sum(x**2-10*np.cos(2*np.pi*x)))
    elif fid == 'f10': return float(-20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
                                    -np.exp(np.mean(np.cos(2*np.pi*x)))+20+np.e)
    elif fid == 'f11': return float(np.sum(x**2)/4000
                                    -np.prod(np.cos(x/np.sqrt(np.arange(1,D+1))))+1)
    elif fid == 'f12':
        w = 1+(x+1)/4
        s = (np.pi/D)*(10*np.sin(np.pi*w[0])**2
             +np.sum((w[:-1]-1)**2*(1+10*np.sin(np.pi*w[1:])**2))+(w[-1]-1)**2)
        return float(s+np.sum(_u(x,10,100,4)))
    elif fid == 'f13':
        s = 0.1*(np.sin(3*np.pi*x[0])**2
                 +np.sum((x[:-1]-1)**2*(1+np.sin(3*np.pi*x[1:])**2))
                 +(x[-1]-1)**2*(1+np.sin(2*np.pi*x[-1])**2))
        return float(s+np.sum(_u(x,5,100,4)))
    raise ValueError(f'Unknown Yao fid: {fid}')

# Métadonnées Table 1 — Yao
YAO_META = [
    ('f1',  'Sphere',            -100,  100,   0.0,  'Unimodal'),
    ('f2',  'Schwefel 2.22',      -10,   10,   0.0,  'Unimodal'),
    ('f3',  'Schwefel 1.2',      -100,  100,   0.0,  'Unimodal'),
    ('f4',  'Schwefel 2.21',     -100,  100,   0.0,  'Unimodal'),
    ('f5',  'Rosenbrock',         -30,   30,   0.0,  'Unimodal'),
    ('f6',  'Step',             -1.28, 1.28,   0.0,  'Unimodal'),
    ('f7',  'Quartic+Noise',     -100,  100,   0.0,  'Unimodal'),
    ('f8',  'Schwefel 2.26',     -500,  500,-418.98, 'Multimodal'),
    ('f9',  'Rastrigin',        -5.12, 5.12,   0.0,  'Multimodal'),
    ('f10', 'Ackley',             -32,   32,   0.0,  'Multimodal'),
    ('f11', 'Griewank',          -600,  600,   0.0,  'Multimodal'),
    ('f12', 'Penalized 1',        -50,   50,   0.0,  'Multimodal'),
    ('f13', 'Penalized 2',        -50,   50,   0.0,  'Multimodal'),
]

# Métadonnées Table 1 — CEC2005 (f14–f27, opfunu officiel)
CEC05_RAW = [
    ('f14','Shifted Sphere',              'F12005', 'Shifted Unimodal'),
    ('f15','Shifted Schwefel 1.2',        'F22005', 'Shifted Unimodal'),
    ('f16','Shifted Rot. Elliptic',       'F32005', 'Shifted Unimodal'),
    ('f17','Shifted Schwefel 1.2+Noise',  'F42005', 'Shifted Unimodal'),
    ('f18','Schwefel 2.6 Bounds',         'F52005', 'Shifted Unimodal'),
    ('f19','Shifted Rosenbrock',          'F62005', 'Shifted Multimodal'),
    ('f20','Shifted Rot. Griewank',       'F72005', 'Shifted Multimodal'),
    ('f21','Shifted Rot. Ackley',         'F82005', 'Shifted Multimodal'),
    ('f22','Shifted Rastrigin',           'F92005', 'Shifted Multimodal'),
    ('f23','Shifted Rot. Rastrigin',      'F102005','Shifted Multimodal'),
    ('f24','Shifted Rot. Weierstrass',    'F112005','Shifted Multimodal'),
    ('f25','Schwefel 2.13',               'F122005','Shifted Multimodal'),
    ('f26','Shifted Exp. Griewank+Ros',   'F132005','Shifted Multimodal'),
    ('f27','Shifted Rot. Scaffer F6',     'F142005','Shifted Multimodal'),
]

CEC05_INST = {}
CEC05_META = []
for fid, name, cls_name, ftype in CEC05_RAW:
    inst = getattr(cec2005, cls_name)(ndim=D)
    CEC05_INST[fid] = inst
    CEC05_META.append((fid, name, float(inst.lb[0]), float(inst.ub[0]),
                        float(inst.f_global), ftype))

# Métadonnées Table 11 — CEC2013 (f28–f55, opfunu officiel)
CEC13_RAW = [
    ('f28','Sphere',                  'F12013',  'Unimodal'),
    ('f29','Rot. High Cond. Elliptic','F22013',  'Unimodal'),
    ('f30','Rot. Bent Cigar',         'F32013',  'Unimodal'),
    ('f31','Rot. Discus',             'F42013',  'Unimodal'),
    ('f32','Different Powers',        'F52013',  'Unimodal'),
    ('f33','Rot. Rosenbrock',         'F62013',  'Multimodal'),
    ('f34','Rot. Schaffer F7',        'F72013',  'Multimodal'),
    ('f35','Rot. Ackley',             'F82013',  'Multimodal'),
    ('f36','Rot. Weierstrass',        'F92013',  'Multimodal'),
    ('f37','Rot. Griewank',           'F102013', 'Multimodal'),
    ('f38','Rastrigin',               'F112013', 'Multimodal'),
    ('f39','Rot. Rastrigin',          'F122013', 'Multimodal'),
    ('f40','Non-Cont. Rot. Rastrigin','F132013', 'Multimodal'),
    ('f41','Schwefel',                'F142013', 'Multimodal'),
    ('f42','Rot. Schwefel',           'F152013', 'Multimodal'),
    ('f43','Rot. Katsuura',           'F162013', 'Multimodal'),
    ('f44','Lunacek Bi-Rastrigin',    'F172013', 'Multimodal'),
    ('f45','Rot. Lunacek Bi-Ras.',    'F182013', 'Multimodal'),
    ('f46','Exp. Griewank+Rosenbrock','F192013', 'Multimodal'),
    ('f47','Exp. Scaffer F6',         'F202013', 'Multimodal'),
    ('f48','Composition 1 (n=5,Rot)', 'F212013', 'Composition'),
    ('f49','Composition 2 (n=3,Unr)', 'F222013', 'Composition'),
    ('f50','Composition 3 (n=3,Rot)', 'F232013', 'Composition'),
    ('f51','Composition 4 (n=3,Rot)', 'F242013', 'Composition'),
    ('f52','Composition 5 (n=3,Rot)', 'F252013', 'Composition'),
    ('f53','Composition 6 (n=5,Rot)', 'F262013', 'Composition'),
    ('f54','Composition 7 (n=5,Rot)', 'F272013', 'Composition'),
    ('f55','Composition 8 (n=5,Rot)', 'F282013', 'Composition'),
]

CEC13_INST = {}
CEC13_META = []
for fid, name, cls_name, ftype in CEC13_RAW:
    inst = getattr(cec2013, cls_name)(ndim=D)
    CEC13_INST[fid] = inst
    CEC13_META.append((fid, name, float(inst.lb[0]), float(inst.ub[0]),
                        float(inst.f_global), ftype))

ALL_META = YAO_META + CEC05_META + CEC13_META

print(f'✓ {len(YAO_META)} fonctions Yao 1999        (f1–f13)  — implémentation native')
print(f'✓ {len(CEC05_META)} fonctions CEC 2005       (f14–f27) — opfunu officiel')
print(f'✓ {len(CEC13_META)} fonctions CEC 2013       (f28–f55) — opfunu officiel')
print(f'   Total : {len(ALL_META)} fonctions')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Unified Evaluator + Utilities
# ═══════════════════════════════════════════════════════════════════

# Index rapide des métadonnées
_META_IDX = {m[0]: m for m in ALL_META}

def evaluate(fid, x):
    if fid in CEC13_INST: return float(CEC13_INST[fid].evaluate(x))
    if fid in CEC05_INST: return float(CEC05_INST[fid].evaluate(x))
    return yao_eval(fid, x)

def get_bounds(fid):
    m = _META_IDX[fid]
    return float(m[2]), float(m[3])

def get_optimum(fid):
    return float(_META_IDX[fid][4])

def fitness_val(fval):
    return 1.0/(1.0+fval) if fval >= 0 else 1.0+abs(fval)

def rand_pop(SN, D, lb, ub):
    return np.random.uniform(lb, ub, (SN, D))

# ── Vérification rapide ──────────────────────────────────────────
print('Vérification évaluateur & optimums :')
for fid in ['f1','f9','f14','f22','f28','f48','f55']:
    lb, ub = get_bounds(fid); opt = get_optimum(fid)
    x = np.random.uniform(lb, ub, D)
    val = evaluate(fid, x)
    print(f'  {fid}: f*={opt:8.1f}  f(x_rand)={val:.4e}  ok={np.isfinite(val)}')
print('✓ Évaluateur OK.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 — ABC standard (Karaboga 2005)
# ═══════════════════════════════════════════════════════════════════

class ABCOptimizer:
    """ABC standard — Karaboga, 2005."""
    def __init__(self, fid, SN, MaxFEs, limit, D):
        self.fid = fid
        self.SN, self.MaxFEs, self.limit, self.D = SN, MaxFEs, limit, D
        self.lb, self.ub = get_bounds(fid)

    def run(self, seed=None):
        if seed is not None: np.random.seed(seed)
        lb, ub, D, SN = self.lb, self.ub, self.D, self.SN
        pop   = rand_pop(SN, D, lb, ub)
        fvals = np.array([evaluate(self.fid, x) for x in pop])
        trial = np.zeros(SN, int)
        best_val = fvals.min()
        NFE = SN
        history = [best_val]

        while NFE < self.MaxFEs:
            # Employed bees
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                k  = np.random.choice([j for j in range(SN) if j != i])
                jj = np.random.randint(D)
                phi = np.random.uniform(-1, 1)
                v = pop[i].copy()
                v[jj] = np.clip(pop[i,jj] + phi*(pop[i,jj]-pop[k,jj]), lb, ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            # Onlooker bees
            fa = np.array([fitness_val(f) for f in fvals])
            pr = fa / (fa.sum() + 1e-300)
            for _ in range(SN):
                if NFE >= self.MaxFEs: break
                i  = np.random.choice(SN, p=pr)
                k  = np.random.choice([j for j in range(SN) if j != i])
                jj = np.random.randint(D)
                phi = np.random.uniform(-1, 1)
                v = pop[i].copy()
                v[jj] = np.clip(pop[i,jj] + phi*(pop[i,jj]-pop[k,jj]), lb, ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            # Scout bees
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                if trial[i] > self.limit:
                    pop[i]   = np.random.uniform(lb, ub, D)
                    fvals[i] = evaluate(self.fid, pop[i]); NFE += 1
                    trial[i] = 0
                    if fvals[i] < best_val: best_val = fvals[i]

            history.append(best_val)

        return best_val, np.array(history)

print('✓ ABCOptimizer (Karaboga 2005) prêt.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 — UMABC (Utility Mining-Guided ABC, proposé)
# Eq.(1) HUFSM · Eq.(2) Anchor-Shift · Eq.(3) Memory-Driven Scout
# ═══════════════════════════════════════════════════════════════════

class UMABCOptimizer:
    """
    Utility Mining-Guided Artificial Bee Colony (UMABC).

    Eq.(1)  U(xi) = α·fit_norm(xi) + (1-α)·dist_norm(xi)
    Eq.(2)  v_{i,j} = x_HUFS,j + φ·(x_{i,j} - x_{k,j})
    Eq.(3)  V_{i,j} = r1·x_bestHUFS + r2·x_rand + r3·(L+r4·(U-L))
    """
    def __init__(self, fid, SN, MaxFEs, limit, D, alpha=0.6, hufs_ratio=0.2):
        self.fid = fid
        self.SN, self.MaxFEs, self.limit, self.D = SN, MaxFEs, limit, D
        self.alpha = alpha
        self.hufs_ratio = hufs_ratio
        self.lb, self.ub = get_bounds(fid)

    def _utility(self, pop, fvals):
        """Eq.(1)"""
        fa = np.array([fitness_val(f) for f in fvals])
        fn = (fa - fa.min()) / (fa.max() - fa.min() + 1e-10)
        d  = np.zeros(self.SN)
        for i in range(self.SN):
            di = np.linalg.norm(pop - pop[i], axis=1); di[i] = np.inf
            d[i] = di.min()
        dn = (d - d.min()) / (d.max() - d.min() + 1e-10)
        return self.alpha * fn + (1 - self.alpha) * dn

    def _hufs(self, pop, fvals):
        utils = self._utility(pop, fvals)
        k     = max(1, int(self.SN * self.hufs_ratio))
        idx   = np.argsort(utils)[::-1][:k]
        return pop[idx], fvals[idx]

    def run(self, seed=None):
        if seed is not None: np.random.seed(seed)
        lb, ub, D, SN = self.lb, self.ub, self.D, self.SN
        pop   = rand_pop(SN, D, lb, ub)
        fvals = np.array([evaluate(self.fid, x) for x in pop])
        trial = np.zeros(SN, int)
        best_val = fvals.min()
        NFE = SN
        history = [best_val]

        while NFE < self.MaxFEs:
            # ── HUFSM — Eq.(1) ─────────────────────────────────────────
            hufs, hufs_fv = self._hufs(pop, fvals)
            x_best = hufs[np.argmin(hufs_fv)]

            # ── Employed — Eq.(2) Anchor-Shift ─────────────────────────
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                jj  = np.random.randint(D)
                v   = pop[i].copy()
                v[jj] = np.clip(anc[jj]+np.random.uniform(-1,1)*(pop[i,jj]-pop[k,jj]),lb,ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            # ── Onlooker — Utility probs + Eq.(2) ──────────────────────
            utils = self._utility(pop, fvals)
            pr    = utils / (utils.sum() + 1e-10)
            for _ in range(SN):
                if NFE >= self.MaxFEs: break
                i   = np.random.choice(SN, p=pr)
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                jj  = np.random.randint(D)
                v   = pop[i].copy()
                v[jj] = np.clip(anc[jj]+np.random.uniform(-1,1)*(pop[i,jj]-pop[k,jj]),lb,ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            # ── Scout — Eq.(3) Memory-Driven Restart ───────────────────
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                if trial[i] > self.limit:
                    r1,r2,r3,r4 = np.random.uniform(0,1,4)
                    xr = pop[np.random.randint(SN)]
                    pop[i] = np.clip(r1*x_best+r2*xr+r3*(lb+r4*(ub-lb)),lb,ub)
                    fvals[i] = evaluate(self.fid, pop[i]); NFE += 1
                    trial[i] = 0
                    if fvals[i] < best_val: best_val = fvals[i]

            history.append(best_val)

        return best_val, np.array(history)

print('✓ UMABCOptimizer prêt — Eq.(1)(2)(3).')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8 — Quick Validation (sanity check before the full run)
# ═══════════════════════════════════════════════════════════════════
print('Sanity check (3 fonctions, 1 run, MaxFEs=500) :')
for fid in ['f1', 'f9', 'f28', 'f48']:
    opt = get_optimum(fid)
    bv_a, _ = ABCOptimizer(fid, 10, 500, 50, D).run(seed=0)
    bv_u, _ = UMABCOptimizer(fid, 10, 500, 50, D).run(seed=0)
    print(f'  {fid}: ABC_err={bv_a-opt:.3e}  UMABC_err={bv_u-opt:.3e}')
print('✓ Sanity check OK.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 9 — MAIN EXPERIMENT
# 30 runs × 55 functions × 2 algorithms (ABC & UMABC)
# Estimated duration: 1–6 h on CPU depending on D
#
# AUTOMATIC RESUME:
#   • If a checkpoint exists, functions already completed are
#     skipped, and calculation resumes from where it stopped.
#   • To force a full restart: results.clear()
# ═══════════════════════════════════════════════════════════════════

# Vérifier combien de fonctions sont déjà terminées
RUNS   = PARAMS['RUNS']
_done  = [fid for fid,r in results.items()
          if len(r.get('abc_err', [])) >= RUNS]
_todo  = [m[0] for m in ALL_META if m[0] not in _done]
print(f'État : {len(_done)}/{len(ALL_META)} fonctions déjà complètes — '
      f'{len(_todo)} restantes.')
if _done:
    print(f'  Complètes : {_done}')

# Démarrer l'autosave
autosave_thread = start_autosave()

SN     = PARAMS['SN']
MaxFEs = PARAMS['MaxFEs']
limit  = PARAMS['limit']
alpha  = PARAMS['alpha']
hr     = PARAMS['HUFS_ratio']

print(f'\nLancement : {len(_todo)} fonctions × {RUNS} runs × 2 algos')
if _todo:
    print(f'Budget NFE restant estimé : {len(_todo)*RUNS*2*MaxFEs:,}')
print()
t0 = time.time()

for meta in ALL_META:
    fid  = meta[0]
    name = meta[1]
    opt  = get_optimum(fid)

    # ── Skip si déjà complet ─────────────────────────────────────
    if len(results.get(fid, {}).get('abc_err', [])) >= RUNS:
        ea = results[fid]['abc_err']
        eu = results[fid]['umabc_err']
        print(f'{fid:4s} {name[:28]:28s}  [déjà complet]  '
              f'ABC={ea.mean():+.3e}±{ea.std():.2e}  '
              f'UMABC={eu.mean():+.3e}±{eu.std():.2e}')
        continue

    # ── Récupérer runs partiels si disponibles ───────────────────
    prev = results.get(fid, {})
    abc_raw   = list(prev.get('abc',   []))
    umabc_raw = list(prev.get('umabc', []))
    h_abc0    = prev.get('abc_hist',   None)
    h_umabc0  = prev.get('umabc_hist', None)
    start_run = len(abc_raw)   # reprend là où ça s'était arrêté

    if start_run > 0:
        print(f'{fid:4s} reprise depuis run {start_run}/{RUNS}...')

    t_fn = time.time()

    for run in range(start_run, RUNS):
        seed = run * 137 + abs(hash(fid)) % 9973

        bv_a, h_a = ABCOptimizer(
            fid, SN, MaxFEs, limit, D).run(seed=seed)
        bv_u, h_u = UMABCOptimizer(
            fid, SN, MaxFEs, limit, D, alpha, hr).run(seed=seed)

        abc_raw.append(bv_a)
        umabc_raw.append(bv_u)
        if run == 0 or h_abc0 is None:
            h_abc0, h_umabc0 = h_a, h_u

        # ── Sauvegarde partielle après chaque run ─────────────────
        # (permet reprise à la granularité d'un run)
        _abc_a_tmp   = np.array(abc_raw)
        _umabc_a_tmp = np.array(umabc_raw)
        _ea_tmp = _abc_a_tmp   - opt
        _eu_tmp = _umabc_a_tmp - opt
        results[fid] = {
            'name'        : name, 'meta': meta, 'optimum': opt,
            'abc'         : _abc_a_tmp,
            'umabc'       : _umabc_a_tmp,
            'abc_err'     : _ea_tmp,
            'umabc_err'   : _eu_tmp,
            'abc_mean'    : _ea_tmp.mean(),  'abc_std'   : _ea_tmp.std(ddof=0),
            'abc_best'    : _ea_tmp.min(),   'abc_worst' : _ea_tmp.max(),
            'abc_median'  : float(np.median(_ea_tmp)),
            'umabc_mean'  : _eu_tmp.mean(),  'umabc_std' : _eu_tmp.std(ddof=0),
            'umabc_best'  : _eu_tmp.min(),   'umabc_worst': _eu_tmp.max(),
            'umabc_median': float(np.median(_eu_tmp)),
            'abc_hist'    : h_abc0,
            'umabc_hist'  : h_umabc0,
        }

    # ── Résumé final pour cette fonction ─────────────────────────
    ea = results[fid]['abc_err']
    eu = results[fid]['umabc_err']
    elapsed = time.time() - t_fn
    delta   = eu.mean() - ea.mean()
    winner  = 'UMABC>' if eu.mean() < ea.mean() else ' ABC> '
    print(f'{fid:4s} {name[:28]:28s}  '
          f'ABC={ea.mean():+.3e}±{ea.std():.2e}  '
          f'UMABC={eu.mean():+.3e}±{eu.std():.2e}  '
          f'Δ={delta:+.3e}  {winner}  [{elapsed:.0f}s]')

stop_autosave()
final_ckpt = manual_save('FINAL')
print(f'\n✓ Expérience terminée en {(time.time()-t0)/3600:.2f}h')
print(f'  Checkpoint final → {final_ckpt}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 10 — Comprehensive Statistical Table (Mean ± Std, Best, Worst, Median)
# Official CEC format — Class A requirement
# ═══════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon, rankdata

def wilcoxon_test(ea, eu, alpha_level=0.05):
    """Test bilatéral sur les erreurs.
    Retourne : (symbole, p-value)
      '+' → UMABC significativement meilleur
      '-' → ABC  significativement meilleur
      '=' → pas de différence significative
    """
    try:
        _, p = wilcoxon(ea, eu)
    except ValueError:
        p = 1.0
    if p < alpha_level:
        return ('+' if eu.mean() < ea.mean() else '-'), p
    return '=', p

rows_stat = []
for meta in ALL_META:
    fid   = meta[0]
    name  = meta[1]
    ftype = meta[5]
    if fid not in results: continue
    r  = results[fid]
    ea = r['abc_err']
    eu = r['umabc_err']
    sig, pval = wilcoxon_test(ea, eu)

    rows_stat.append({
        'Fn'          : fid,
        'Name'        : name,
        'Type'        : ftype,
        # ABC
        'ABC_best'    : r['abc_best'],
        'ABC_worst'   : r['abc_worst'],
        'ABC_median'  : r['abc_median'],
        'ABC_mean'    : r['abc_mean'],
        'ABC_std'     : r['abc_std'],
        # UMABC
        'UMABC_best'  : r['umabc_best'],
        'UMABC_worst' : r['umabc_worst'],
        'UMABC_median': r['umabc_median'],
        'UMABC_mean'  : r['umabc_mean'],
        'UMABC_std'   : r['umabc_std'],
        # Test statistique
        'Sig'         : sig,
        'p_value'     : pval,
        # Amélioration relative (%)
        'Improve_%'   : (r['abc_mean']-r['umabc_mean'])/(abs(r['abc_mean'])+1e-30)*100,
    })

df = pd.DataFrame(rows_stat)

# ── W/T/L global ─────────────────────────────────────────────────
W = (df['Sig']=='+').sum()
T = (df['Sig']=='=').sum()
L = (df['Sig']=='-').sum()
print(f'W/T/L global (UMABC vs ABC) : {W} / {T} / {L}  sur {len(df)} fonctions')
print()

# ── W/T/L par groupe ─────────────────────────────────────────────
for grp in df['Type'].unique():
    sub = df[df['Type']==grp]
    w = (sub['Sig']=='+').sum()
    t = (sub['Sig']=='=').sum()
    l = (sub['Sig']=='-').sum()
    print(f'  {grp:22s}: W={w:2d} T={t:2d} L={l:2d}')

# ── Affichage condensé ───────────────────────────────────────────
print()
disp = df[['Fn','Name','ABC_mean','ABC_std','UMABC_mean','UMABC_std',
           'Improve_%','p_value','Sig']].copy()
for c in ['ABC_mean','ABC_std','UMABC_mean','UMABC_std']:
    disp[c] = disp[c].map('{:.3e}'.format)
disp['Improve_%'] = disp['Improve_%'].map('{:+.1f}%'.format)
disp['p_value']   = disp['p_value'].map('{:.4f}'.format)
print(tabulate(disp, headers='keys', tablefmt='grid', showindex=False))

# Sauvegarde
df.to_csv('results_ABC_UMABC/benchmark_full_stats.csv', index=False)
print('\n✓ Sauvegardé → results_ABC_UMABC/benchmark_full_stats.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 11 — Official CEC Error Table
# Best / Worst / Median / Mean / Std — Over 30 runs
# ═══════════════════════════════════════════════════════════════════

print('TABLEAU ERREURS |f(x)-f*| — FORMAT CEC OFFICIEL')
print('='*80)

err_rows = []
for meta in ALL_META:
    fid   = meta[0]; name = meta[1]; ftype = meta[5]
    if fid not in results: continue
    r = results[fid]
    for algo, pfx in [('ABC','abc'),('UMABC','umabc')]:
        e = r[f'{pfx}_err']
        err_rows.append({
            'Fn': fid, 'Name': name[:30], 'Type': ftype, 'Algo': algo,
            'Best'  : e.min(),
            'Worst' : e.max(),
            'Median': np.median(e),
            'Mean'  : e.mean(),
            'Std'   : e.std(ddof=0),
        })

df_err = pd.DataFrame(err_rows)
df_err.to_csv('results_ABC_UMABC/error_cec_format.csv', index=False)

# Affichage UMABC
u_only = df_err[df_err['Algo']=='UMABC'][['Fn','Name','Best','Worst','Median','Mean','Std']].copy()
for c in ['Best','Worst','Median','Mean','Std']:
    u_only[c] = u_only[c].map('{:.3e}'.format)
print('UMABC :')
print(tabulate(u_only, headers='keys', tablefmt='grid', showindex=False))
print('\n✓ Sauvegardé → results_ABC_UMABC/error_cec_format.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 12 — Friedman Average Rank (Classe A)
# ═══════════════════════════════════════════════════════════════════

r_abc, r_umabc = [], []
for meta in ALL_META:
    fid = meta[0]
    if fid not in results: continue
    r = results[fid]
    rk = rankdata([r['abc_mean'], r['umabc_mean']])  # 1 = meilleur
    r_abc.append(rk[0])
    r_umabc.append(rk[1])

avg_abc   = np.mean(r_abc)
avg_umabc = np.mean(r_umabc)

print('═'*55)
print('CLASSEMENT FRIEDMAN MOYEN (plus bas = meilleur)')
print(f'  ABC   : {avg_abc:.4f}')
print(f'  UMABC : {avg_umabc:.4f}')
winner = 'UMABC' if avg_umabc < avg_abc else 'ABC'
print(f'  → Vainqueur : {winner}')
print('═'*55)

print('\nPar groupe :')
groups_idx = {
    'Yao Unimodal'   : [f'f{i}' for i in range(1, 8)],
    'Yao Multimodal' : [f'f{i}' for i in range(8, 14)],
    'CEC2005'        : [f'f{i}' for i in range(14, 28)],
    'CEC13 Unimodal' : [f'f{i}' for i in range(28, 33)],
    'CEC13 Multi'    : [f'f{i}' for i in range(33, 48)],
    'CEC13 Compo'    : [f'f{i}' for i in range(48, 56)],
}
for grp, fids in groups_idx.items():
    sub = df[df['Fn'].isin(fids)]
    if sub.empty: continue
    w = (sub['Sig']=='+').sum()
    t = (sub['Sig']=='=').sum()
    l = (sub['Sig']=='-').sum()
    print(f'  {grp:18s} : W={w:2d}  T={t:2d}  L={l:2d}')

pd.DataFrame({
    'Algorithm'       : ['ABC','UMABC'],
    'Avg Friedman Rank': [avg_abc, avg_umabc],
    'Wins_W'          : [L, W],
    'Ties_T'          : [T, T],
    'Losses_L'        : [W, L],
}).to_csv('results_ABC_UMABC/friedman_summary.csv', index=False)
print('\n✓ Sauvegardé → results_ABC_UMABC/friedman_summary.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 13 — Tables LaTeX 
# Table 1 style (f1–f27) + 3 tables CEC2013 (f28–f55)
# Colonnes : Fn | Mean ± Std (ABC) | Mean ± Std (UMABC) | Improve% | Sig
# ═══════════════════════════════════════════════════════════════════

def fmt_sci(v, prec=2):
    if v == 0 or not np.isfinite(v): return r'$0$'
    m, e = f'{v:.{prec}e}'.split('e')
    return f'${m}\\times10^{{{int(e)}}}$'

def sig_sym(s):
    return {'+'  : r'$\bullet$',
            '-'  : r'$\circ$',
            '='  : r'$\approx$'}.get(s, r'$\approx$')

def make_latex(subdf, caption, label):
    lines = [
        r'\begin{table*}[htbp]',
        r'\centering',
        r'\caption{' + caption + '}',
        r'\label{' + label + '}',
        r'\setlength{\tabcolsep}{3.5pt}',
        r'\begin{tabular}{llrrrrrr}',
        r'\toprule',
        (r'Fn & Function & \multicolumn{2}{c}{ABC (Karaboga 2005)} & '
         r'\multicolumn{2}{c}{UMABC (Proposed)} & $\Delta\%$ & Sig \\'),
        r'\cmidrule(lr){3-4}\cmidrule(lr){5-6}',
        r' & & Mean $\pm$ Std & & Mean $\pm$ Std & & & \\',
        r'\midrule',
    ]

    prev_type = None
    for _, row in subdf.iterrows():
        # Séparateur de groupe
        if row['Type'] != prev_type:
            lines.append(r'\midrule')
            lines.append(r'\multicolumn{8}{l}{\textit{' + row['Type'] + r'}} \\')
            prev_type = row['Type']

        bold = (row['Sig'] == '+')
        bo, bc = (r'\textbf{', '}') if bold else ('', '')

        abc_str   = f"{fmt_sci(row['ABC_mean'])} $\\pm$ {fmt_sci(row['ABC_std'])}"
        umabc_str = f"{bo}{fmt_sci(row['UMABC_mean'])} $\\pm$ {fmt_sci(row['UMABC_std'])}{bc}"

        lines.append(
            f"{row['Fn']} & {row['Name'][:30]} & "
            f"\\multicolumn{{2}}{{c}}{{{abc_str}}} & "
            f"\\multicolumn{{2}}{{c}}{{{umabc_str}}} & "
            f"{row['Improve_%']:+.1f} & {sig_sym(row['Sig'])} \\\\"
        )

    W_ = (subdf['Sig']=='+').sum()
    T_ = (subdf['Sig']=='=').sum()
    L_ = (subdf['Sig']=='-').sum()
    lines += [
        r'\midrule',
        r'\multicolumn{8}{l}{W/T/L (UMABC vs ABC): ' + f'{W_}/{T_}/{L_}' + r'} \\',
        r'\bottomrule',
        r'\end{tabular}',
        r'\begin{tablenotes}\small',
        (r'\item $\bullet$: UMABC sig. better; $\circ$: ABC sig. better; '
         r'$\approx$: no significant diff. (Wilcoxon $\alpha$=0.05, bilateral).'),
        (r'\item SN=' + str(PARAMS['SN']) +
         r', MaxFEs=' + str(PARAMS['MaxFEs']) +
         r', D=' + str(PARAMS['D']) +
         r', Runs=' + str(PARAMS['RUNS']) +
         r', $\alpha_U=' + str(PARAMS['alpha']) +
         r'$, HUFS ratio=' + str(PARAMS['HUFS_ratio']) + r'.'),
        r'\end{tablenotes}',
        r'\end{table*}',
    ]
    return '\n'.join(lines)


# ── Table 1 (f1–f27) ─────────────────────────────────────────────
df_t1 = df[df['Fn'].apply(lambda x: int(x[1:]) <= 27)]
tex   = make_latex(df_t1,
    f'Comparison results on Table~1 benchmark functions (D={D}, 30 independent runs)',
    'tab:table1')
with open('results_ABC_UMABC/latex_table1.tex','w') as f: f.write(tex)
print('✓ results_ABC_UMABC/latex_table1.tex')

# ── Table 11 — CEC2013, 3 sous-tables ────────────────────────────
df_t11 = df[df['Fn'].apply(lambda x: int(x[1:]) >= 28)]
for grp in ['Unimodal','Multimodal','Composition']:
    sub = df_t11[df_t11['Type'] == grp]
    if sub.empty: continue
    tex = make_latex(sub,
        f'CEC~2013 {grp} functions (Table~11): UMABC vs ABC (D={D}, 30 runs)',
        f'tab:cec2013_{grp.lower()}')
    path = f'results_ABC_UMABC/latex_table11_{grp.lower()}.tex'
    with open(path,'w') as f: f.write(tex)
    print(f'✓ {path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 14 — Convergence Curves (7×4 Grids)
# (f1–f27) and (f28–f55)
# ═══════════════════════════════════════════════════════════════════

def convergence_grid(fn_list, title, path, ncols=4):
    fn_list = [f for f in fn_list if f in results]
    nrows   = (len(fn_list) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.6, nrows*2.8))
    axes = axes.flatten()

    for idx, fid in enumerate(fn_list):
        r   = results[fid]
        opt = r['optimum']
        ax  = axes[idx]
        for key, color, ls, lbl in [
            ('abc_hist',   C_ABC,   '--', 'ABC'),
            ('umabc_hist', C_UMABC, '-',  'UMABC'),
        ]:
            h   = r[key]
            x   = np.linspace(0, PARAMS['MaxFEs'], len(h))
            err = np.abs(h - opt) + 1e-12
            ax.semilogy(x, err, color=color, lw=1.3, ls=ls, label=lbl)
        ax.set_title(f'{fid}: {r["name"][:22]}', fontsize=7, pad=2)
        ax.set_xlabel('NFE', fontsize=6)
        ax.set_ylabel('$|f(x)-f^*|$', fontsize=7)
        ax.tick_params(labelsize=6)

    for idx in range(len(fn_list), len(axes)):
        axes[idx].set_visible(False)

    handles = [
        Line2D([0],[0],color=C_ABC,  lw=1.5,ls='--',label='ABC (Karaboga 2005)'),
        Line2D([0],[0],color=C_UMABC,lw=1.5,ls='-', label='UMABC (Proposed)'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=9,
               bbox_to_anchor=(0.5,-0.01), frameon=True)
    fig.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print(f'✓ {path}')

convergence_grid(
    [f'f{i}' for i in range(1, 28)],
    f'Convergence curves — Table 1 (f1–f27), D={D}',
    'results_ABC_UMABC/fig1_convergence_table1.pdf'
)
convergence_grid(
    [f'f{i}' for i in range(28, 56)],
    f'Convergence curves — Table 11 CEC2013 (f28–f55), D={D}',
    'results_ABC_UMABC/fig2_convergence_table11.pdf'
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 15 — Box Plots par groupe (erreurs |f(x)-f*|)
# ═══════════════════════════════════════════════════════════════════

bp_props = dict(
    patch_artist=True, notch=False,
    medianprops=dict(color='k', lw=1.5),
    whiskerprops=dict(lw=0.8),
    capprops=dict(lw=0.8),
    flierprops=dict(marker='o', ms=2.5, alpha=0.5),
)

groups_bp = [
    ('Yao Unimodal',   [f'f{i}' for i in range(1, 8)],   'fig3a'),
    ('Yao Multimodal', [f'f{i}' for i in range(8, 14)],  'fig3b'),
    ('CEC2005',        [f'f{i}' for i in range(14, 28)],  'fig3c'),
    ('CEC13 Uni',      [f'f{i}' for i in range(28, 33)],  'fig3d'),
    ('CEC13 Multi',    [f'f{i}' for i in range(33, 48)],  'fig3e'),
    ('CEC13 Compo',    [f'f{i}' for i in range(48, 56)],  'fig3f'),
]

for gname, fids, tag in groups_bp:
    fids = [f for f in fids if f in results]
    if not fids: continue
    n   = len(fids)
    fig, axes = plt.subplots(1, n, figsize=(n*2.0+0.5, 4))
    if n == 1: axes = [axes]

    for ax, fid in zip(axes, fids):
        r  = results[fid]
        ea = r['abc_err']
        eu = r['umabc_err']
        bp = ax.boxplot([ea, eu], labels=['ABC','UMABC'], **bp_props)
        for patch, c in zip(bp['boxes'], [C_ABC, C_UMABC]):
            patch.set_facecolor(c); patch.set_alpha(0.55)
        # Ajouter Mean comme '+'
        for xi, vals in zip([1, 2], [ea, eu]):
            ax.plot(xi, vals.mean(), marker='+', color='white',
                    ms=8, mew=1.5, zorder=5)
        ax.set_title(fid, fontsize=8)
        ax.set_ylabel('$f(x)-f^*$', fontsize=7)
        ax.tick_params(labelsize=7)

    legend_patches = [
        mpatches.Patch(color=C_ABC,   label='ABC'),
        mpatches.Patch(color=C_UMABC, label='UMABC'),
        Line2D([0],[0],marker='+',color='k',ls='',ms=8,label='Mean'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3,
               fontsize=8, bbox_to_anchor=(0.5,-0.05))
    fig.suptitle(f'Error distribution — {gname} (30 runs, D={D})', fontsize=10)
    plt.tight_layout()
    path = f'results_ABC_UMABC/{tag}_boxplot_{gname.replace(" ","_")}.pdf'
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print(f'✓ {path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 16 — Graphe d'amélioration (%) avec marqueurs Wilcoxon
# Séparé : Table 1 (f1–f27) et Table 11 (f28–f55)
# ═══════════════════════════════════════════════════════════════════

def improvement_bar(fn_list, title, path):
    sub  = df[df['Fn'].isin(fn_list)].copy()
    fids = sub['Fn'].tolist()
    imps = sub['Improve_%'].tolist()
    sigs = sub['Sig'].tolist()
    grps = sub['Type'].tolist()

    # Couleurs par type fonctionnel
    type_colors = {
        'Unimodal'        : '#4C8BF5',
        'Multimodal'      : C_UMABC,
        'Shifted Unimodal': '#8E44AD',
        'Shifted Multimodal':'#E67E22',
        'Composition'     : '#C0392B',
    }
    clrs = [type_colors.get(g, C_TIE) for g in grps]

    fig, ax = plt.subplots(figsize=(max(14, len(fids)*0.55), 5))
    ax.bar(fids, imps, color=clrs, edgecolor='white', lw=0.4)
    ax.axhline(0, color='k', lw=0.8)

    for i, (v, s) in enumerate(zip(imps, sigs)):
        if s != '=':
            ax.text(i, v + (0.8 if v >= 0 else -0.8), '★',
                    ha='center', va='bottom' if v >= 0 else 'top',
                    fontsize=10, color='k')

    ax.set_xlabel('Benchmark function', fontsize=11)
    ax.set_ylabel('Improvement of UMABC over ABC (%)', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(len(fids)))
    ax.set_xticklabels(fids, rotation=55, ha='right', fontsize=8)

    patches = [mpatches.Patch(color=c, label=t)
               for t, c in type_colors.items()]
    patches.append(mpatches.Patch(color='white',
                   label='★ p<0.05 (Wilcoxon)', ec='k'))
    ax.legend(handles=patches, fontsize=8, loc='best', ncol=2)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print(f'✓ {path}')

improvement_bar(
    [f'f{i}' for i in range(1, 28)],
    f'UMABC vs ABC — % Improvement on Table 1 (f1–f27), D={D}',
    'results_ABC_UMABC/fig4a_improvement_table1.pdf'
)
improvement_bar(
    [f'f{i}' for i in range(28, 56)],
    f'UMABC vs ABC — % Improvement on Table 11 CEC2013 (f28–f55), D={D}',
    'results_ABC_UMABC/fig4b_improvement_table11.pdf'
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 17 — Radar Chart — profil normalisé 55 fonctions
# ═══════════════════════════════════════════════════════════════════

all_fids_ord = [f'f{i}' for i in range(1, 56) if f'f{i}' in results]
N = len(all_fids_ord)

abc_sc, umabc_sc = [], []
for fid in all_fids_ord:
    r  = results[fid]
    ea = abs(r['abc_mean'])   + 1e-30
    eu = abs(r['umabc_mean']) + 1e-30
    tot = ea + eu
    abc_sc.append(1 - ea/tot)      # plus haut = meilleur
    umabc_sc.append(1 - eu/tot)

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
for lst in [abc_sc, umabc_sc, angles]:
    lst.append(lst[0])

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.plot(angles, abc_sc,   color=C_ABC,   lw=1.5, ls='--', label='ABC')
ax.fill(angles, abc_sc,   color=C_ABC,   alpha=0.12)
ax.plot(angles, umabc_sc, color=C_UMABC, lw=1.8, label='UMABC (Proposed)')
ax.fill(angles, umabc_sc, color=C_UMABC, alpha=0.18)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(all_fids_ord, fontsize=6)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_title(
    f'Normalized performance profile — 55 functions, D={D}\n(higher = better)',
    pad=25, fontsize=11)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('results_ABC_UMABC/fig5_radar_55fn.pdf', bbox_inches='tight')
plt.show()
print('✓ results_ABC_UMABC/fig5_radar_55fn.pdf')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 18 — Sensibilité de α (UMABC)
# 9 valeurs × 7 fonctions représentatives × 10 runs
# ═══════════════════════════════════════════════════════════════════

ALPHA_VALS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SENS_FNS   = ['f1','f9','f12','f16','f28','f35','f48']  # 1 par groupe
SENS_RUNS  = 10   # ← augmenter à 30 pour soumission finale

alpha_mean = {fid: [] for fid in SENS_FNS}
alpha_std  = {fid: [] for fid in SENS_FNS}

for av in ALPHA_VALS:
    for fid in SENS_FNS:
        opt  = get_optimum(fid)
        errs = []
        for run in range(SENS_RUNS):
            v, _ = UMABCOptimizer(
                fid, PARAMS['SN'], PARAMS['MaxFEs'],
                PARAMS['limit'], D, alpha=av).run(seed=run*7)
            errs.append(v - opt)
        alpha_mean[fid].append(np.mean(errs))
        alpha_std[fid].append(np.std(errs))
    print(f'  α={av:.1f} ✓')

fig, ax = plt.subplots(figsize=(7, 4))
ls_map = ['-','--','-.',':','-','--','-.']
mk_map = ['o','s','^','D','v','P','X']
for i, fid in enumerate(SENS_FNS):
    v = np.array(alpha_mean[fid])
    rng = np.ptp(v)
    vn  = (v - v.min()) / (rng + 1e-30)
    ax.plot(ALPHA_VALS, vn, ls=ls_map[i], marker=mk_map[i],
            ms=5, lw=1.3, label=fid)

ax.axvline(PARAMS['alpha'], color='red', lw=1.2, ls='--', alpha=0.7,
           label=f'Default α={PARAMS["alpha"]}')
ax.set_xlabel('α  (utility balance parameter)', fontsize=11)
ax.set_ylabel('Normalized mean error (lower = better)', fontsize=11)
ax.set_title(f'Sensitivity analysis of α — UMABC, D={D}', fontsize=11)
ax.legend(fontsize=9, ncol=2)
ax.set_xticks(ALPHA_VALS)
plt.tight_layout()
plt.savefig('results_ABC_UMABC/fig6_sensitivity_alpha.pdf', bbox_inches='tight')
plt.show()
print('✓ results_ABC_UMABC/fig6_sensitivity_alpha.pdf')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 19 — Scalabilité (D=50 et D=100, runs réduits)
# Obligatoire pour publication Classe A
# ═══════════════════════════════════════════════════════════════════
# Décommenter les lignes ci-dessous pour l'exécuter
# (désactivé par défaut pour éviter un runtime trop long)

SCALABILITY_DIMS  = [50, 100]   # dimensions à tester
SCALABILITY_RUNS  = 30          # 30 runs — Classe A
# Fonctions représentatives (1 par groupe)
SCALE_FNS = ['f1','f3','f9','f11','f14','f22','f28','f35','f48']

scale_results = {}

for Ds in SCALABILITY_DIMS:
    MaxFEs_s = 5_000 * Ds
    SN_s     = Ds
    # Re-instancier opfunu avec la nouvelle dimension
    _cec05_s = {}
    for fid, _, cls_name, _ in CEC05_RAW:
        if fid in SCALE_FNS:
            _cec05_s[fid] = getattr(cec2005, cls_name)(ndim=Ds)
    _cec13_s = {}
    for fid, _, cls_name, _ in CEC13_RAW:
        if fid in SCALE_FNS:
            _cec13_s[fid] = getattr(cec2013, cls_name)(ndim=Ds)

    def eval_s(fid, x):
        if fid in _cec13_s: return float(_cec13_s[fid].evaluate(x))
        if fid in _cec05_s: return float(_cec05_s[fid].evaluate(x))
        return yao_eval(fid, x)

    for fid in SCALE_FNS:
        lb_s, ub_s = get_bounds(fid)
        opt_s      = get_optimum(fid)
        abc_e, umabc_e = [], []
        for run in range(SCALABILITY_RUNS):
            seed = run*137 + abs(hash(fid+str(Ds))) % 9973
            np.random.seed(seed)
            # ABC
            pop = np.random.uniform(lb_s, ub_s, (SN_s, Ds))
            fvals = np.array([eval_s(fid, x) for x in pop])
            trial = np.zeros(SN_s, int); NFE = SN_s; bv = fvals.min()
            while NFE < MaxFEs_s:
                for i in range(SN_s):
                    if NFE >= MaxFEs_s: break
                    k = np.random.choice([j for j in range(SN_s) if j!=i])
                    jj = np.random.randint(Ds)
                    v = pop[i].copy()
                    v[jj] = np.clip(pop[i,jj]+np.random.uniform(-1,1)*(pop[i,jj]-pop[k,jj]),lb_s,ub_s)
                    fv = eval_s(fid,v); NFE+=1
                    if fv<=fvals[i]: pop[i]=v;fvals[i]=fv;trial[i]=0
                    else: trial[i]+=1
                    if fv<bv: bv=fv
                fa=np.array([fitness_val(f) for f in fvals]); pr=fa/(fa.sum()+1e-300)
                for _ in range(SN_s):
                    if NFE>=MaxFEs_s: break
                    i=np.random.choice(SN_s,p=pr); k=np.random.choice([j for j in range(SN_s) if j!=i])
                    jj=np.random.randint(Ds); v=pop[i].copy()
                    v[jj]=np.clip(pop[i,jj]+np.random.uniform(-1,1)*(pop[i,jj]-pop[k,jj]),lb_s,ub_s)
                    fv=eval_s(fid,v);NFE+=1
                    if fv<=fvals[i]: pop[i]=v;fvals[i]=fv;trial[i]=0
                    else: trial[i]+=1
                    if fv<bv: bv=fv
                for i in range(SN_s):
                    if NFE>=MaxFEs_s: break
                    if trial[i]>PARAMS['limit']:
                        pop[i]=np.random.uniform(lb_s,ub_s,Ds)
                        fvals[i]=eval_s(fid,pop[i]);NFE+=1;trial[i]=0
                        if fvals[i]<bv: bv=fvals[i]
            abc_e.append(bv - opt_s)
            # UMABC — appel direct de la classe (réutilise eval_s via monkey-patch)
            orig_inst = (CEC13_INST.get(fid), CEC05_INST.get(fid))
            if fid in _cec13_s: CEC13_INST[fid] = _cec13_s[fid]
            if fid in _cec05_s: CEC05_INST[fid] = _cec05_s[fid]
            bv_u, _ = UMABCOptimizer(fid, SN_s, MaxFEs_s,
                PARAMS['limit'], Ds, PARAMS['alpha'], PARAMS['HUFS_ratio']).run(seed=seed)
            # Restaurer
            if orig_inst[0] is not None: CEC13_INST[fid]=orig_inst[0]
            if orig_inst[1] is not None: CEC05_INST[fid]=orig_inst[1]
            umabc_e.append(bv_u - opt_s)

        ea = np.array(abc_e); eu = np.array(umabc_e)
        scale_results[(fid, Ds)] = {
            'abc_mean':ea.mean(),'abc_std':ea.std(),
            'umabc_mean':eu.mean(),'umabc_std':eu.std(),
        }
        print(f'  D={Ds} {fid}: ABC={ea.mean():.3e}±{ea.std():.2e}  '
              f'UMABC={eu.mean():.3e}±{eu.std():.2e}')

# Tableau scalabilité
scale_rows = []
for (fid, Ds), r in scale_results.items():
    scale_rows.append({'Fn':fid, 'D':Ds,
        'ABC Mean':r['abc_mean'],  'ABC Std':r['abc_std'],
        'UMABC Mean':r['umabc_mean'],'UMABC Std':r['umabc_std']})
df_scale = pd.DataFrame(scale_rows)
df_scale.to_csv('results_ABC_UMABC/scalability_D50_D100.csv', index=False)
print('✓ Sauvegardé → results_ABC_UMABC/scalability_D50_D100.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 20 — Packaging final ZIP
# ═══════════════════════════════════════════════════════════════════
import zipfile

# Sauvegarde pickle final
final_pkl = manual_save('EXPERIMENT_COMPLETE')

zip_path = 'UMABC_vs_ABC_full_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(glob.glob('results_ABC_UMABC/*.*')):
        zf.write(f)
    for f in sorted(glob.glob('results_ABC_UMABC/checkpoints/chk_FINAL*.pkl')):
        zf.write(f)
    for f in sorted(glob.glob('results_ABC_UMABC/checkpoints/chk_EXPERIMENT*.pkl')):
        zf.write(f)

print(f'✓ Archive → {zip_path}')
print('Contenu :')
for f in sorted(glob.glob('results_ABC_UMABC/*.*')):
    print(f'  {f:65s}  ({os.path.getsize(f)/1024:.1f} Ko)')

try:
    from google.colab import files
    files.download(zip_path)
    print('\n↓ Téléchargement Colab lancé.')
except ImportError:
    print(f'\n(Local : archive disponible dans {os.path.abspath(zip_path)})')

---
## Récapitulatif des livrables générés

| Fichier | Description |
|---------|-------------|
| `benchmark_full_stats.csv` | Tous résultats : Mean±Std, Best, Worst, Median, Wilcoxon, p-value, Δ% |
| `error_cec_format.csv` | Format CEC officiel (Best/Worst/Median/Mean/Std) par algo |
| `friedman_summary.csv` | Rang Friedman moyen + W/T/L par groupe |
| `scalability_D50_D100.csv` | Scalabilité D=50 et D=100 (Mean±Std) |
| `latex_table1.tex` | Table LaTeX IEEE — f1–f27 (Yao+CEC2005), Mean±Std |
| `latex_table11_unimodal.tex` | Table LaTeX — CEC2013 Unimodal |
| `latex_table11_multimodal.tex` | Table LaTeX — CEC2013 Multimodal |
| `latex_table11_composition.tex` | Table LaTeX — CEC2013 Composition |
| `fig1_convergence_table1.pdf` | Courbes convergence f1–f27 |
| `fig2_convergence_table11.pdf` | Courbes convergence f28–f55 |
| `fig3a–f_boxplot_*.pdf` | Box plots (Mean+Std visible) par groupe |
| `fig4a_improvement_table1.pdf` | Δ% UMABC>ABC, Table 1, ★=Wilcoxon sig. |
| `fig4b_improvement_table11.pdf` | Δ% UMABC>ABC, Table 11, ★=Wilcoxon sig. |
| `fig5_radar_55fn.pdf` | Radar chart 55 fonctions |
| `fig6_sensitivity_alpha.pdf` | Sensibilité α |
| `checkpoints/chk_FINAL_*.pkl` | Checkpoint pickle compressé |

---
### Checklist Classe A — Complète

- [x] **2 algorithmes** : ABC (Karaboga 2005) vs UMABC (proposé) — NABC retiré
- [x] **Mean ± Std** calculé sur 30 runs (ddof=0, conforme CEC)
- [x] **Best / Worst / Median / Mean / Std** exporté (format CEC officiel)
- [x] **55 fonctions** : f1–f13 Yao + f14–f27 CEC2005 + f28–f55 CEC2013
- [x] **opfunu** — shifts/rotations officiels chargés des fichiers binaires CEC
- [x] **30 runs** par fonction, graine déterministe par run
- [x] **Wilcoxon signed-rank** bilatéral sur erreurs |f(x)−f*|, α=0.05
- [x] **Friedman rank** moyen global et par groupe
- [x] **Scalabilité** D=50 et D=100 (Cell 19)
- [x] **Tables LaTeX** IEEE/Elsevier — Mean±Std, W/T/L, bullet/circ, groupes typographiés
- [x] **Sensibilité α** — 9 valeurs sur 7 fonctions représentatives
- [x] **Autosave 30 min** — pickle compressé + CSV partiel, reprise sur interruption
- [x] **300 DPI, Times New Roman** — toutes figures
